# First example - Tracer breakthrough

In this lesson, we will:
- Create and connect our first systems of unit operations.
- Run CADET and analyze the results.

## Example 1: Our target flow sheet.

In a first example, we will look at a simple system with just three unit operations: an [Inlet](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Inlet.html), a Column (modeled as an [LRM](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.LumpedRateModelWithoutPores.html)), and an [Outlet](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Outlet.html).

```{figure} ./resources/flow_sheet_PFR.png
:width: 60%
```

We will load the column with a non-binding tracer molecule (such as e.g. Dextran or Micromer particles) and inspect the shape and timing of the breakthrough curve.

## 1. Setting up the model

The common denominator of everything we model (unit operations, flows, events, etc.) are the components that we model. So let us first create a `ComponentSystem` to manage and keep track of our components.

#### Component System

- `ComponentSystem` ensure that all parts of the process have the same number of components.
- Components can be named which automatically adds legends to the plot methods.

For advanced use, see [here](https://cadet-process.readthedocs.io/en/latest/reference/process_model/component_system.html).

In [ ]:
from CADETProcess.processModel import ComponentSystem

component_system = ComponentSystem(1)

## Unit Operations

For an overview of all models in CADET-Process, see [here](https://cadet-process.readthedocs.io/en/v0.11.1/reference/processModel.html#unit-operation-models).

Unit operations require:
- the `ComponentSystem`
- as a unique name.

Note that the name string passed in the constructor is later used to reference the unit in the flow sheet for setting `Events` and `OptimizationVariables`.

## Inlet

In CADET, the `Inlet` pseudo unit operation serves as source for the system and is used to create arbitary concentration profiles as boundary conditions (see also [here](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Inlet.html#CADETProcess.)processModel.Inlet).

- Concentration profiles are described using a third degree piecewise polynomial for each component.
- Flow rate can be expressed as a third degree piecewise polynomial.

Here, the flow rate is constant, we can directly set the parameter on the object.

Note that we have to convert all units to SI units.


```{note}
Generally, CADET can be used with any consistent system of units.
However, we strongly recommend converting everything to the SI system.
```

In [ ]:
from CADETProcess.processModel import Inlet

inlet = Inlet(component_system, "inlet")
inlet.c = 1  # mol / m^3
inlet.flow_rate = 1e-7  # m^3 / s

Note that every unit operation model has different model parameters.
To display all parameters, simply print the `parameters` attribute.

In [ ]:
print(inlet.parameters)

## Outlet
The `Outlet` is another pseudo unit operation that serves as sink for the system (see also [here](https://cadet-process.readthedocs.io/en/v0.11.1/reference/generated/CADETProcess.processModel.Outlet.html#CADETProcess.processModel.Outlet)).

In [ ]:
from CADETProcess.processModel import Outlet
outlet = Outlet(component_system, "outlet")

Note that the `Outlet` unit does not have any model parameters.

In [ ]:
print(outlet.parameters)

## Column: Lumped Rate Model without Pores

In this example, we will use the `LumpedRateModelWithoutPores`. For the model equations see [here](https://cadet.github.io/master/modelling/unit_operations/lumped_rate_model_without_pores.html) and the parameters [here](https://cadet.github.io/v5.1.0/interface/unit_operations/lumped_rate_model_without_pores.html#lumped-rate-model-without-pores-config).

Assume the following parameters:


| Parameter        | Value | Unit          |
| ---------------- | ----- | ------------- |
| Length           | 0.1   | $m$           |
| Diameter         | 0.2   | $m$           |
| Porosity         | 0.4   | -             |
| Axial Dispersion | 1e-7  | $m^2 s^{-1}$  |

In [ ]:
from CADETProcess.processModel import LumpedRateModelWithoutPores

column = LumpedRateModelWithoutPores(component_system, "column")
print(column.required_parameters)

In [ ]:
column.length = 0.1

In [ ]:
print(column.missing_parameters)

Let's fill in the missing parameters analogously:

In [ ]:
column.diameter = 0.2
column.total_porosity = 0.4
column.axial_dispersion = 1e-7

## Flow Sheet Connectivity

```{figure} ./resources/flow_sheet_PFR.png
:width: 60%
```

The `FlowSheet` stores the connectivity between different unit operations.

For more information, see also [here](https://cadet-process.readthedocs.io/en/latest/reference/process_model/flow_sheet.html).

In [ ]:
from CADETProcess.processModel import FlowSheet

flow_sheet = FlowSheet(component_system, "flow_sheet")

flow_sheet.add_unit(inlet)
flow_sheet.add_unit(outlet)

Let's add the column analogously:

In [ ]:
flow_sheet.add_unit(column)

Now all unit operations are in the flow sheet, but we need to connect them:

In [ ]:
flow_sheet.add_connection(inlet, column)

Let's connect the column to the outlet as well:

In [ ]:
flow_sheet.add_connection(column, outlet)

## The Process

The Process class handles all time-dependent configurations. Since we have no dynamic events, we just need to set the cycle time.

In [ ]:
from CADETProcess.processModel import Process
process = Process(flow_sheet, "process")
process.cycle_time = 30*60  # s

## 3. Setting up the simulator and running the simulation

To simulate the process, a process simulator needs to be configured.
If no path is specified, CADET-Process will try to autodetect CADET.

In [ ]:
from CADETProcess.simulator import Cadet

simulator = Cadet()

If a specific version of CADET is to be used, add the install path to the constructor:

```
process_simulator = Cadet(install_path='/path/to/cadet/executable')
```

To check that everything works correctly, you can call the check_cadet method:

In [ ]:
simulator.check_cadet()

Now, run the simulation:

In [ ]:
simulation_results = simulator.simulate(process)

## 4. Plotting the results

The simulation_results object contains the solution for every unit operation.
```{note}
By default, solutions for the inlet and the outlet of all unit operations are stored. While this is mostly relevant for unit operations that contain a transformative step in a process, inlets and outlets are written as well with identical inlets and outlets.
```

Solutions for all UOs are stored as a solution objects inside a nested dictionary, that can be accessed via dot notation:

In [ ]:
simulation_results.solution

The solutions of the inlet UO are stored in

In [ ]:
simulation_results.solution.inlet

and the solution object of intereset can be accessed:

In [ ]:
simulation_results.solution.inlet.outlet

From this the raw solution array can be read out

In [ ]:
simulation_results.solution.inlet.outlet.solution

or convinient in-built plotting methods can be used:

### Inlet Profile

In [ ]:
_ = simulation_results.solution.inlet.outlet.plot()

### Outlet Profile

Equivilantly, the outlet profile can be plot:

In [ ]:
_ = simulation_results.solution.outlet.outlet.plot()

## Visualization

In addition to the solution at the inlet and outlet of a unit operation, we can also take a look inside the column to see the peak move.

For this purpose, set the flag in the unit's `SolutionRecorder`.
Then, the `SimulationResults` will also contain an entry for the bulk.

```{note} 
Since this solution is two-dimensinal (space and time), the solution can be plotted at a given position (`plot_at_location`) or a given time (`plot_at_time`).
```

In [ ]:
column.solution_recorder.write_solution_bulk = True

simulation_results = simulator.simulate(process)

In [ ]:
simulation_results.solution.column.bulk.plot_at_position(0.05)
simulation_results.solution.column.bulk.plot_at_time(80)

With ipywidgets the propagation of concentration over the whole column can be visualized for all time steps:

In [ ]:
%matplotlib widget
from ipywidgets import IntSlider, interactive_output
import ipywidgets as widgets
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


try:
    ui.close()
except Exception:
    pass
try:
    plt.close(fig)
except Exception:
    pass

# Fresh figure
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_ylim(0, 1)
plt.tight_layout()

def draw(time=0):
    ax.cla()
    simulation_results.solution.column.bulk.plot_at_time(time, ax=ax)
    ax.set_ylim(0, 1)
    fig.canvas.draw_idle()

time_slider = IntSlider(
    min=0,
    max=int(process.cycle_time),
    step=10,
    layout={'width': '800px'},
    description='Time',
    style={'description_width': 'initial'},
)

ui = interactive_output(draw, {'time': time_slider})

display(time_slider, ui)
draw(0)  